# 03 · Retrieval-augmented generation

**Session 01 · notebook 3 of 6**

## Objective

Connect the vector store to the model: retrieve the passages that match the question and supply them as context. See two retrieval failures, their fixes, and why retrieval and generation are evaluated separately.

## The application after this notebook

```
question ──> retriever ──> top-k passages ──┐
                                             ├──> [ instructions + passages + question ] ──> model ──> answer with citations
question ────────────────────────────────────┘
```

The chain is written in LangChain's expression language: a retrieval step feeding a prompt feeding the model.

In [1]:
# Setup. In Colab this clones the repository and installs the dependencies; locally it uses the folder as is.
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/avsolatorio/ai4data.git"   # replace with the published repository when released
BRANCH = "feat/training-series"
SESSION = "training/session-01-building-ai-applications/notebooks"

if not pathlib.Path("workshop.py").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, "/content/ai4data"], check=True)
    os.chdir(f"/content/ai4data/{SESSION}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, os.getcwd())

# Optional: a key for an OpenAI-compatible API. Without one, the notebook replays recorded outputs and says so.
try:
    from google.colab import userdata  # type: ignore
    for name in ("WORKSHOP_API_KEY", "WORKSHOP_BASE_URL", "WORKSHOP_MODEL"):
        if userdata.get(name):
            os.environ[name] = userdata.get(name)
except Exception:
    pass

import workshop as w
model = w.make_model()
print("Model:", w.mode(model))

Model: recorded outputs (no WORKSHOP_API_KEY set)


In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

docs = w.load_documents()
chunks = w.split_sections(docs)
store = w.build_store(chunks)
print(len(chunks), "chunks in the store")

INSTRUCTIONS = """You are a research assistant for development data.
Answer only from the supplied passages. After each figure, cite the passage in the square-bracket form given.
If the passages do not contain what is needed, say so."""

prompt = ChatPromptTemplate.from_messages([("system", INSTRUCTIONS), ("human", "PASSAGES:\n{context}\n\nQUESTION: {question}")])


def make_rag(k=4, where=None):
    """A RAG chain: question -> retrieved passages -> prompt -> model."""
    retrieve = RunnableLambda(lambda q: store.similarity_search(q, k=k, filter=where))
    return {"context": retrieve | w.format_context, "question": RunnablePassthrough()} | prompt | model


def hits(question, k=4, where=None):
    return store.similarity_search_with_score(question, k=k, filter=where)

21 chunks in the store


## 1. The worked example through the pipeline

Retrieve, then answer. The hits are printed first so that the evidence the model saw is visible.

In [3]:
w.print_hits(hits(w.MAIN_QUESTION, k=4))
print()
w.use("nb03_rag_main")
w.show(make_rag(k=4).invoke(w.MAIN_QUESTION))

 0.763  [rwanda-digital · Internet use · 2026]
 0.748  [regional-2016 · intro · 2016]
 0.676  [kenya-digital · Internet use · 2026]
 0.591  [mobile-money · intro · 2026]

[recorded output: nb03_rag_main]
Internet use. Rwanda's share of individuals using the internet reached 31.7 percent of the
population in 2024, up from 8.0 percent in 2010 [rwanda-digital · Internet use · 2026]. Kenya's
reached 35.0 percent in 2024, up from 7.2 percent in 2010 [kenya-digital · Internet use · 2026]. A
2016 summary gives 16.6 percent for Kenya and 12.5 percent for Rwanda in 2015 and expected about 25
percent by 2020 [regional-2016 · intro · 2016]; the later briefs show that expectation was met in
Kenya and nearly met in Rwanda.

Economic development. The supplied passages contain no figures on GDP or economic growth, so that
half of the question cannot be answered from them. A passage on mobile money notes that it runs on
basic phones and that its use differs from internet use [mobile-money · intro · 20

## 2. A retrieval failure: a question with two parts

The question asks about internet access and economic development. One query embeds both, and the internet passages win every slot: the economy briefs are absent from the top four, and the model says that half of the question cannot be answered. The 2016 summary also ranks high, for a reason the next failure makes explicit.

The fix is on the retrieval side: ask the store two questions and merge the passages. (LangChain's `MultiQueryRetriever` does this with a model generating the sub-queries; here the two are written by hand so the mechanism is visible.)

In [4]:
sub_queries = ["internet use in Kenya and Rwanda, recent figures", "GDP per capita and economic growth in Kenya and Rwanda, recent figures"]
merged = []
for q in sub_queries:
    for d in store.similarity_search(q, k=3):
        if d not in merged:
            merged.append(d)
w.print_hits(merged)
print()
w.use("nb03_rag_two_queries")
w.show((prompt | model).invoke({"context": w.format_context(merged), "question": w.MAIN_QUESTION}))

[regional-2016 · intro · 2016]
[rwanda-digital · Internet use · 2026]
[kenya-digital · Internet use · 2026]
[rwanda-economy · Reading the two indicators together · 2026]
[kenya-economy · GDP per capita · 2026]
[kenya-economy · Reading the two indicators together · 2026]

[recorded output: nb03_rag_two_queries]
Internet use. Kenya's share of individuals using the internet reached 35.0 percent of the population
in 2024, up from 7.2 percent in 2010 [kenya-digital · Internet use · 2026]. Rwanda's reached 31.7
percent in 2024, up from 8.0 percent in 2010 [rwanda-digital · Internet use · 2026].

Economic development. Kenya's GDP per capita in current US dollars was 2,133 in 2024, after a dip to
1,943 in 2023 [kenya-economy · GDP per capita · 2026]. Rwanda's was 1,060 in 2024, having passed
1,000 in 2022 [rwanda-economy · GDP per capita · 2026]. Rwanda recorded growth above 7 percent in
most years [rwanda-economy · Reading the two indicators together · 2026].

Patterns. Kenya has higher level

## 3. A retrieval failure: outdated evidence

"What are the latest available figures for internet use in Kenya and Rwanda?" The 2016 summary contains the words "latest available figures" and ranks first. With two passages supplied, the Kenya figure comes from 2016 and the Rwanda figure from 2026, and the model reports both faithfully.

In [5]:
q = "What are the latest available figures for internet use in Kenya and Rwanda?"
w.print_hits(hits(q, k=2))
print()
w.use("nb03_rag_outdated")
w.show(make_rag(k=2).invoke(q))

 0.816  [regional-2016 · intro · 2016]
 0.765  [rwanda-digital · Internet use · 2026]

[recorded output: nb03_rag_outdated]
Kenya: the latest available figure in the passages is 16.6 percent of the population in 2015, up
from 7.2 percent in 2010; the summary notes that later values will be added as released
[regional-2016 · intro · 2016]. Rwanda: 31.7 percent in 2024, up from 28.8 percent in 2023 [rwanda-
digital · Internet use · 2026].


The answer is faithful to its evidence and wrong for the reader: a 2015 figure presented as the latest. Nothing in the generation step can fix this; the evidence was selected badly.

## 4. The fix is on the retrieval side

Three options:

1. **A metadata filter**: exclude documents dated before 2020 for a question about the latest figures. `InMemoryVectorStore` takes a filter function over the document metadata.
2. **Query reformulation**: search for what the passage would say ("internet use 2024 percent of population"). It helps in many cases and fails in this one, as the cell shows: the 2016 summary still ranks first, because its wording is close to any query about figures.
3. **Re-ranking by date** after retrieval, when the store has no filter.

The filter is the reliable fix here, because the date is metadata the embedding does not see.

In [6]:
since_2020 = lambda d: str(d.metadata.get("date", "")) >= "2020"
print("1. metadata filter (date >= 2020):")
w.print_hits(hits(q, k=2, where=since_2020))
print("\n2. reformulated query:")
w.print_hits(hits("Kenya and Rwanda internet use 2024, percent of population", k=2))
print()
w.use("nb03_rag_filtered")
w.show(make_rag(k=2, where=since_2020).invoke(q))

1. metadata filter (date >= 2020):
 0.765  [rwanda-digital · Internet use · 2026]
 0.703  [kenya-digital · Internet use · 2026]

2. reformulated query:
 0.871  [regional-2016 · intro · 2016]
 0.833  [rwanda-digital · Internet use · 2026]



[recorded output: nb03_rag_filtered]
Kenya: 35.0 percent of the population used the internet in 2024, up from 32.1 percent in 2023
[kenya-digital · Internet use · 2026]. Rwanda: 31.7 percent in 2024, up from 28.8 percent in 2023
[rwanda-digital · Internet use · 2026]. Both figures are World Development Indicators values for
IT.NET.USER.ZS, as reproduced in the briefs.


## 5. Retrieval always returns something

A question outside the collection still gets top-k passages, ranked by whatever is least unlike it. Compare a question the collection can answer with one it cannot.

In [7]:
for q2 in ["What is the population of Rwanda?", "What was inflation in Uganda in 2023?"]:
    print(q2)
    w.print_hits(hits(q2, k=3))
    print()

What is the population of Rwanda?
 0.739  [rwanda-digital · Population · 2026]
 0.607  [rwanda-economy · Reading the two indicators together · 2026]
 0.595  [rwanda-economy · GDP per capita · 2026]

What was inflation in Uganda in 2023?
 0.534  [rwanda-economy · GDP per capita · 2026]
 0.523  [kenya-economy · GDP per capita · 2026]
 0.503  [rwanda-economy · Reading the two indicators together · 2026]



Both questions get three passages with a score. The out-of-collection question scores lower, and a threshold could separate the two here, but a threshold has to be set from examples on the embedding model in use, which is part of the evaluation in notebook 06. The second safeguard is the instruction that tells the model to say when the passages do not answer the question, which notebook 01 introduced. Applications use both.

## 6. Retrieval and generation are evaluated separately

Retrieval can be tested without calling a model: for each evaluation question, did the top-k passages include a passage from the expected document? The cell runs that test for all eight questions at two values of k.

In [8]:
questions = w.load_questions()
print(f"{'id':4s} {'category':18s} {'k=2':6s} {'k=4':6s} expected documents")
for qd in questions:
    r2 = w.retrieval_hit(store.similarity_search(qd["question"], k=2), qd["expected_documents"])
    r4 = w.retrieval_hit(store.similarity_search(qd["question"], k=4), qd["expected_documents"])
    print(f"{qd['id']:4s} {qd['category']:18s} {'hit' if r2 else 'miss':6s} {'hit' if r4 else 'miss':6s} {', '.join(qd['expected_documents'])}")

id   category           k=2    k=4    expected documents


q1   main               hit    hit    kenya-digital, rwanda-digital, kenya-economy, rwanda-economy
q2   factual            hit    hit    rwanda-digital
q3   ambiguous          hit    hit    kenya-digital
q4   missing-data       hit    hit    kenya-digital


q5   out-of-scope       hit    hit    coffee
q6   factual            hit    hit    kenya-economy, rwanda-economy


q7   definition         hit    hit    internet-indicator
q8   outdated-evidence  hit    hit    kenya-digital, rwanda-digital


A miss here is a retrieval problem, fixed by chunking, metadata, ranking, or the query. A wrong answer with a hit is a generation problem, fixed by instructions or a different model. Mixing the two makes both impossible to improve.

## Key points

- Retrieval selects the evidence; the model writes from it. A faithful answer to the wrong evidence is wrong.
- Fixes for retrieval live in metadata, ranking, and query formulation. The model cannot repair a bad selection.
- Retrieval always returns something; a threshold set from examples and an instruction to say "no evidence" are the two safeguards.
- Test retrieval on its own, with expected documents per question.

## Exercise

Change `w.split_sections` to `w.split_fixed` in the first cell and rerun section 6. Count the misses. Then try k=8.

## Where this connects to the program

- [Practical Guide to Serving Official Statistics to AI Agents](https://worldbank.github.io/ai4data/cookbook/serving-statistics-to-agents/): the design of tools over a statistical catalog, in full.
- [Practical Guide to Evaluation Suites for Statistical AI](https://worldbank.github.io/ai4data/cookbook/evaluation-suites/): how to build and run the tests.
- [Proof-Carrying Numbers](https://worldbank.github.io/ai4data/docs/pcn/): the claim-tag protocol the verifier in this session follows.
- [Data360 MCP server](https://github.com/worldbank/data360-mcp): the tools of notebook 04, exposed to any MCP client over the World Bank's data platform.
- [End-to-end reference implementation](https://worldbank.github.io/ai4data/docs/reference-implementation/): documentation, validation, publication, agent access, and verification in one run.